# 01 — Chuẩn bị CNN TensorFlow
Giữ kiến trúc và thiết lập thực nghiệm cũ: AdamW, cosine LR, batch64, augmentation OpenCV và BatchNorm tương ứng.
Chỉ đọc train/dev. Bộ kết quả dùng trong slide nằm ở `../results_224`: C04, epoch25, Accuracy97.36%.
Chạy lại sẽ lưu riêng trong `runs/CNN` hoặc Output Kaggle. Kết quả có thể khác do môi trường và tính ngẫu nhiên.

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_CNN_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_CNN/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('CNN', PROJECT_ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import display, Image
from src.CNN.data import CLASSES, CANDIDATES, load_config, configure_gpu, seed_everything
configure_gpu()
seed_everything(42)

## 1. Lập manifest và khai báo thực nghiệm

In [ ]:
from src.CNN.data import prepare
protocol, manifests = prepare(DATA_ROOT, RESULTS_ROOT)
display(pd.read_csv(RESULTS_ROOT/'inputs/image_counts.csv'))
display(pd.DataFrame(protocol['candidates']))

## 2. Đọc, resize và minh họa HSV/HOG
Ảnh vẫn có giá trị 0–255; lớp Rescaling trong CNN chia 255 đúng một lần.
HSV/HOG ở đây dùng để đối chiếu ảnh với KNN/SVM, **không phải đầu vào feature của CNN**.

In [ ]:
from src.experiments.figures import input_figures, hsv_histograms
input_figures(DATA_ROOT, manifests['train'], RESULTS_ROOT/'inputs/figures', 'CNN')
hsv_histograms(DATA_ROOT, manifests['train'], RESULTS_ROOT/'inputs/figures', 'CNN')

## 3. Augmentation chỉ trên train

In [ ]:
from src.CNN.analysis import augmentation_examples
augmentation_examples(DATA_ROOT, RESULTS_ROOT)

## 4. Thiết lập và cách chọn checkpoint
Batch64; AdamW lr0.001, weight_decay0.0001, beta=(0.9,0.999), epsilon1e-8; clip gradient toàn cục norm5.
Cosine theo epoch tương đương CosineAnnealingLR(T_max=25, eta_min=0.00001); tối đa25 epoch, patience6 theo dev Macro-F1/MAE.
Xáo trộn toàn bộ danh sách đường dẫn mỗi epoch; chỉ train có augmentation. GPU tự bật mixed_float16 khi train; chia255 và BatchNorm tính float32, headsoftmax/lossfloat32. Dev/test suy luận float32 như mã PyTorch cũ.
Không pretrained, không chọn theo test. Khác RNG/kernel giữa hai thư viện vẫn có thể làm kết quả khác; kết quả chạy lại có thể khác lượt đã công bố.
Save Version thành công rồi gắn Output đó vào02.